# Notebook 07 — Module 7: Reproducible software and portfolio
**Goal:** turn Notebooks 01–06 into a public, reproducible GitHub repository: an installable Python package, tests, CI, documentation and a concise scientific report. Every number in the report and README is read from your saved result files, so nothing is typed by hand.

**What this notebook does**
1. Creates the repository skeleton, copies the library modules your notebooks already wrote, and **patches one bug** found while writing the tests (see below).
2. Writes new library modules (`annotation`, `features`, `targets`, `config`, `demo`, `pipeline`, `cli`) — the notebook logic refactored into functions.
3. Writes a **test suite** (brute-force checks, the reach guarantee, leakage rules) and runs it.
4. Runs the **synthetic end-to-end demo** (works for anyone, no data needed).
5. **Equivalence check on your real data:** the refactored functions must reproduce the features and targets stored by Notebooks 03–04.
6. Captures the exact environment, curates results and figures, cleans copies of your notebooks, and audits the repo (large files, secrets).
7. Generates `reports/scientific_report.md`, `README.md`, `LICENSE`, `CITATION.cff`, makes a local git commit, and zips the repo.

**A bug caught by the new tests.** The reach test (delete every cell beyond 30 µm and check the features do not change) exposed a latent bug in `nearest_type_distance` from Notebook 03: a cell type with exactly **one** cell gave NaN instead of the true distance. It does not affect your results (every type in your data has thousands of cells; step 5 confirms this), but the library is now correct and has a regression test.

**Before running:** edit the four variables in the next code cell (name, e-mail, GitHub username).

In [ ]:
import os, sys, json, re, shutil, subprocess, platform, importlib, time, warnings
import importlib.metadata as imd
from pathlib import Path
import numpy as np, pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/spatial_ml_breast_cancer")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./spatial_ml_breast_cancer")).resolve()

# ------------------------- EDIT THESE -------------------------
AUTHOR = "Your Name"
AUTHOR_EMAIL = "you@example.com"
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "spatial-ml-xenium-breast"
RUN_EQUIVALENCE = True          # recompute features/targets with the library and compare with Notebooks 03-04 (a few minutes)
# --------------------------------------------------------------

REPO_DIR = PROJECT_DIR / REPO_NAME
TAB_DIR, FIG_DIR, REP_DIR, PROC_DIR = (PROJECT_DIR / p for p in ["results/tables", "results/figures", "reports", "data/processed"])
for d in ["src/spatialml", "tests", "docs", "scripts", "configs", "notebooks", "results/tables", "results/figures", "reports", ".github/workflows"]:
    (REPO_DIR / d).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)                       # %%writefile cells below write relative to the repository root

missing = [p for p in ["configs/config.json", "reports/notebook_03_report.json"] if not (PROJECT_DIR / p).exists()]
print("Project:", PROJECT_DIR, "\nRepository:", REPO_DIR, "\nMissing earlier outputs:", missing or "none")

Mounted at /content/drive
Project: /content/drive/MyDrive/spatial_ml_breast_cancer 
Repository: /content/drive/MyDrive/spatial_ml_breast_cancer/spatial-ml-xenium-breast 
Missing earlier outputs: none


## Part 1 — Reuse the modules your notebooks wrote (single source of truth) and patch the bug
`neighborhood`, `stats`, `modeling` and `evaluation` were generated by Notebooks 03–06 and produced your results, so the repository copies them unchanged — except for the corrected `nearest_type_distance`.

In [ ]:
BUGFIX = '''def nearest_type_distance(xy, type_codes, type_index):
    \"\"\"Distance from each cell to the nearest OTHER cell of the given type (NaN if there is no such cell).\"\"\"
    mask = type_codes == type_index
    n = int(mask.sum())
    if n == 0:
        return np.full(len(xy), np.nan)
    t = cKDTree(xy[mask])
    if n == 1:                                   # a single cell of this type: every other cell can still measure to it
        d, _ = t.query(xy, k=1, workers=-1)
        return np.where(mask, np.nan, d)
    d, _ = t.query(xy, k=2, workers=-1)
    return np.where(mask, d[:, 1], d[:, 0])
'''.replace('\\"', '"')

SRC_DIR = PROJECT_DIR / "src/spatialml"
for m in ["neighborhood", "stats", "modeling", "evaluation"]:
    assert (SRC_DIR / f"{m}.py").exists(), f"missing {m}.py - run Notebooks 03-06 first"
    text = (SRC_DIR / f"{m}.py").read_text()
    if m == "neighborhood" and "a single cell of this type" not in text:
        start = text.index("def nearest_type_distance"); end = text.index("\n\n\ndef ", start)
        text = text[:start] + BUGFIX.rstrip("\n") + text[end:]
        print("Patched nearest_type_distance (single-cell-type bug).")
    (REPO_DIR / f"src/spatialml/{m}.py").write_text(text)
print("Copied modules:", sorted(p.name for p in (REPO_DIR / "src/spatialml").glob("*.py")))

Patched nearest_type_distance (single-cell-type bug).
Copied modules: ['evaluation.py', 'modeling.py', 'neighborhood.py', 'stats.py']


## Part 2 — New library modules (the notebook logic, refactored into tested functions)

In [ ]:
%%writefile src/spatialml/__init__.py
"""spatialml - leakage-safe spatial machine learning for spatial transcriptomics (Xenium)."""
__version__ = "1.0.0"

from . import neighborhood, stats, modeling, evaluation, annotation, features, targets, config, demo, pipeline  # noqa: F401


Writing src/spatialml/__init__.py


In [ ]:
%%writefile src/spatialml/config.py
"""spatialml.config - default configuration, loading, validation and seeding."""
import copy
import json
import random
from pathlib import Path

import numpy as np

DEFAULT_CONFIG = {
    "seed": 42,
    "coordinate_units": "micron",
    "features": {"radii_um": [15, 30], "cap_um": 30, "knn_ks": [5, 10]},
    "target": {"annulus_um": [50, 100], "min_cells_in_annulus": 10, "k": 3},
    "cv": {"block_um": 500, "buffer_um": 100, "folds": 5},
    "model": {"learning_rate": 0.05, "max_leaf_nodes": 15, "max_iter": 200,
              "l2_regularization": 1.0, "min_samples_leaf": 100},
    "evaluation": {"n_bootstrap": 1000, "max_train_cells": 80000},
}


def _merge(base, over):
    out = copy.deepcopy(base)
    for k, v in (over or {}).items():
        out[k] = _merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def validate_config(cfg):
    """The leakage rules of the project, enforced in code."""
    f, t, cv = cfg["features"], cfg["target"], cfg["cv"]
    if max(f["radii_um"]) > f["cap_um"]:
        raise ValueError("feature radii must not exceed the reach cap")
    if f["cap_um"] >= t["annulus_um"][0]:
        raise ValueError("feature reach cap must be smaller than the inner edge of the target ring")
    if cv["buffer_um"] < t["annulus_um"][1]:
        raise ValueError("CV buffer must be at least the outer radius of the target ring")
    return cfg


def load_config(path=None, overrides=None):
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    if path is not None:
        if not Path(path).exists():
            raise FileNotFoundError(f"configuration file not found: {path}")
        cfg = _merge(cfg, json.load(open(path)))
    return validate_config(_merge(cfg, overrides))


def save_config(cfg, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    json.dump(cfg, open(path, "w"), indent=2)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    return np.random.default_rng(seed)


Writing src/spatialml/config.py


In [ ]:
%%writefile src/spatialml/annotation.py
"""spatialml.annotation - marker-based cluster annotation with explicit uncertainty, and documented label refinement."""
import numpy as np
import pandas as pd
import scipy.sparse as sp

COARSE_TYPES = ["Epithelial", "T_cell", "B_cell", "Myeloid", "Mast_cell", "Endothelial", "Mural", "Stromal"]

LINEAGE_TO_COARSE = {
    "Epithelial_luminal": "Epithelial", "Epithelial_basal_luminal": "Epithelial", "Myoepithelial_basal": "Epithelial",
    "T_cell": "T_cell", "B_cell": "B_cell", "Mast_cell": "Mast_cell", "Endothelial": "Endothelial",
    "Macrophage_myeloid": "Myeloid", "Dendritic_cell": "Myeloid", "Myeloid": "Myeloid",
    "Pericyte_smooth_muscle": "Mural", "Mural_myoepithelial": "Mural",
    "Fibroblast_stromal": "Stromal", "Stromal_adipose": "Stromal", "Adipocyte": "Stromal",
}

# An ambiguous cluster is resolved ONLY when its top two lineages are a biologically compatible pair.
COMPATIBLE_PAIRS = [
    ({"Macrophage_myeloid", "Dendritic_cell"}, "Myeloid"),
    ({"Myoepithelial_basal", "Pericyte_smooth_muscle"}, "Mural_myoepithelial"),
    ({"Myoepithelial_basal", "Epithelial_luminal"}, "Epithelial_basal_luminal"),
    ({"Adipocyte", "Fibroblast_stromal"}, "Stromal_adipose"),
]


def cluster_mean_expression(expr, clusters, gene_names):
    """Mean expression per cluster. expr: (cells x genes) dense or sparse; returns (DataFrame, sizes Series)."""
    cl = pd.Categorical(np.asarray(clusters).astype(str))
    codes, k, n = cl.codes, len(cl.categories), expr.shape[0]
    ind = sp.csr_matrix((np.ones(n, dtype=np.float32), (codes, np.arange(n))), shape=(k, n))
    sums = ind @ expr
    sums = sums.toarray() if sp.issparse(sums) else np.asarray(sums)
    sizes = np.bincount(codes, minlength=k)
    return pd.DataFrame(sums / sizes[:, None], index=cl.categories, columns=list(gene_names)), pd.Series(sizes, index=cl.categories, name="n_cells")


def annotate_clusters(means, sizes, markers, min_top=0.35, amb_ratio=0.80):
    """Score each cluster for each lineage (min-max normalised mean marker expression); flag ambiguous clusters."""
    present = {k: [g for g in v if g in means.columns] for k, v in markers.items()}
    present = {k: v for k, v in present.items() if len(v) >= 2}
    span = means.max(axis=0) - means.min(axis=0)
    norm = (means - means.min(axis=0)) / span.where(span > 0.05, np.inf)
    scores = pd.DataFrame({lin: norm[g].mean(axis=1) for lin, g in present.items()})
    arr = scores.to_numpy(); order = np.argsort(-arr, axis=1); r = np.arange(len(scores))
    table = pd.DataFrame({"n_cells": sizes, "top_lineage": scores.columns[order[:, 0]], "top_score": arr[r, order[:, 0]],
                          "second_lineage": scores.columns[order[:, 1]], "second_score": arr[r, order[:, 1]]}, index=scores.index)
    table["score_ratio"] = table["second_score"] / table["top_score"].clip(lower=1e-9)
    table["confident"] = (table["top_score"] >= min_top) & (table["score_ratio"] < amb_ratio)
    table["label"] = np.where(table["confident"], table["top_lineage"], "Ambiguous")
    return table, scores


def refine_labels(table, min_top=0.40):
    """Resolve ambiguous clusters that are compatible lineage pairs; everything else becomes 'Unassigned'."""
    fine, rule = [], []
    for _, row in table.iterrows():
        if bool(row["confident"]):
            fine.append(row["label"]); rule.append("confident")
            continue
        pair = {row["top_lineage"], row["second_lineage"]}
        for members, name in COMPATIBLE_PAIRS:
            if pair == members and row["top_score"] >= min_top:
                fine.append(name); rule.append("resolved: compatible pair"); break
        else:
            fine.append("Unassigned"); rule.append("kept ambiguous")
    out = table.copy(); out["fine_label"] = fine; out["rule"] = rule
    return out


def to_coarse(fine_labels):
    return pd.Series(fine_labels).map(LINEAGE_TO_COARSE).fillna("Unassigned").to_numpy()


Writing src/spatialml/annotation.py


In [ ]:
%%writefile src/spatialml/features.py
"""spatialml.features - leakage-safe spatial neighbourhood features.

Every feature depends only on cells within `cap_um` of the focal cell (self excluded). Nearest-type distances are
CAPPED at cap_um: an unbounded distance would encode cells far outside the feature radius, including the target ring.
"""
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from .annotation import COARSE_TYPES
from .neighborhood import nearest_type_distance, ring_stats, shannon_entropy

UNASSIGNED = "Unassigned"


def type_codes(cell_type, types=COARSE_TYPES):
    all_types = list(types) + [UNASSIGNED]
    lut = {t: i for i, t in enumerate(all_types)}
    codes = pd.Series(np.asarray(cell_type, dtype=object)).map(lut).fillna(len(all_types) - 1).astype(int).to_numpy()
    return codes, all_types


def _rname(r):
    return int(r) if float(r).is_integer() else r


def build_spatial_features(xy, cell_type, radii=(15, 30), cap_um=30, knn_ks=(5, 10), types=COARSE_TYPES, chunk=20000):
    """Return a DataFrame of strict spatial features (one row per cell, in input order)."""
    xy = np.asarray(xy, dtype=np.float64)
    if max(radii) > cap_um:
        raise ValueError("radii must not exceed cap_um")
    codes, all_types = type_codes(cell_type, types)
    onehot = np.eye(len(all_types), dtype=np.float32)[codes]
    stats = ring_stats(xy, onehot, [(0, r) for r in radii], chunk=chunk)
    feat = {}
    for r in radii:
        st = stats[(0, r)]; n = st["n"].astype(np.float64); cnt = st["counts"]; rn = _rname(r)
        feat[f"n_cells_r{rn}"] = n
        feat[f"density_r{rn}"] = n / (np.pi * (r / 1000.0) ** 2)
        with np.errstate(divide="ignore", invalid="ignore"):
            frac = cnt / n[:, None]
        frac[n == 0] = np.nan
        for k, t in enumerate(all_types):
            feat[f"frac_{t}_r{rn}"] = frac[:, k]
        asg = cnt[:, :-1]; na = asg.sum(1)
        feat[f"entropy_r{rn}"] = shannon_entropy(asg)
        with np.errstate(divide="ignore", invalid="ignore"):
            dom = asg.max(1) / na
            edge = np.hypot(st["vx"], st["vy"]) / n
            mnd = st["dsum"] / n
        dom[na == 0] = np.nan; edge[n == 0] = np.nan; mnd[n == 0] = np.nan
        feat[f"dominant_frac_r{rn}"] = dom
        feat[f"edge_asymmetry_r{rn}"] = edge
        feat[f"mean_nbr_dist_r{rn}"] = mnd
    tree = cKDTree(xy)
    kmax = max(knn_ks)
    dk, _ = tree.query(xy, k=kmax + 1, workers=-1)
    dk = np.minimum(dk.reshape(len(xy), -1)[:, 1:], cap_um)          # cap EACH neighbour distance (missing -> inf -> cap)
    for k in knn_ks:
        feat[f"knn_dist_cap{_rname(cap_um)}_k{k}"] = dk[:, :k].mean(1)
    for i, t in enumerate(types):
        d = nearest_type_distance(xy, codes, i)
        feat[f"dist_nearest_{t}_cap{_rname(cap_um)}"] = np.where(np.isnan(d), cap_um, np.minimum(d, cap_um))
    return pd.DataFrame(feat)


def reach_test(xy, cell_type, n_cells=30, seed=0, **kw):
    """Perturbation test of the reach guarantee: delete every cell farther than cap_um from a focal cell and recompute
    its features. Returns the largest absolute difference (should be ~0)."""
    xy = np.asarray(xy, dtype=np.float64); ct = np.asarray(cell_type, dtype=object)
    cap = kw.get("cap_um", 30)
    full = build_spatial_features(xy, ct, **kw)
    tree = cKDTree(xy); rng = np.random.default_rng(seed)
    worst = 0.0
    for i in rng.choice(len(xy), min(n_cells, len(xy)), replace=False):
        idx = np.array(sorted(tree.query_ball_point(xy[i], cap + 1e-9)))
        sub = build_spatial_features(xy[idx], ct[idx], **kw)
        a = full.iloc[i].to_numpy(float); b = sub.iloc[int(np.where(idx == i)[0][0])].to_numpy(float)
        if np.any(np.isnan(a) != np.isnan(b)):
            return float("inf")
        ok = ~np.isnan(a)
        if ok.any():
            worst = max(worst, float(np.abs(a[ok] - b[ok]).max()))
    return worst


Writing src/spatialml/features.py


In [ ]:
%%writefile src/spatialml/targets.py
"""spatialml.targets - microenvironment target from the composition of an annulus around each cell."""
import numpy as np
from scipy.spatial import cKDTree  # noqa: F401  (kept for API symmetry)
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

from .annotation import COARSE_TYPES
from .features import type_codes
from .neighborhood import ring_stats


def annulus_composition(xy, cell_type, annulus=(50, 100), types=COARSE_TYPES, min_cells=10, chunk=20000):
    """Composition (fractions of assigned cells) in the ring r_in <= d <= r_out, self excluded.
    Returns (frac (n x T) with NaN where invalid, n_cells_in_ring, valid mask)."""
    codes, all_types = type_codes(cell_type, types)
    onehot = np.eye(len(all_types), dtype=np.float32)[codes]
    ring = (annulus[0], annulus[1])
    st = ring_stats(np.asarray(xy, float), onehot, [ring], chunk=chunk)[ring]
    cnt = st["counts"]; n_ann = st["n"]; n_asg = cnt[:, :-1].sum(1)
    valid = (n_ann >= min_cells) & (n_asg >= min_cells)
    with np.errstate(divide="ignore", invalid="ignore"):
        frac = cnt[:, :-1] / n_asg[:, None]
    frac[~valid] = np.nan
    return frac, n_ann, valid


def hellinger(frac):
    return np.sqrt(np.nan_to_num(frac))


def fit_target_classes(frac, valid, k=3, seed=42, n_init=10):
    """k-means on Hellinger-transformed composition. Returns raw labels (-1 where invalid)."""
    H = hellinger(frac); vidx = np.where(valid)[0]
    km = KMeans(k, n_init=n_init, random_state=seed).fit(H[vidx])
    labels = np.full(len(frac), -1); labels[vidx] = km.labels_
    return labels


def name_classes(frac, labels, types=COARSE_TYPES):
    """Name classes from their mean composition; numbered by size (ME1 = largest)."""
    ks = sorted(int(c) for c in np.unique(labels) if c >= 0)
    means = {c: np.nanmean(frac[labels == c], axis=0) for c in ks}
    sizes = {c: int((labels == c).sum()) for c in ks}
    names = {}
    for rank, c in enumerate(sorted(ks, key=lambda c: -sizes[c]), start=1):
        o = np.argsort(-means[c]); t1, t2 = types[o[0]], types[o[1]]
        names[c] = f"ME{rank}_{t1}-rich" if means[c][o[0]] >= 0.55 else f"ME{rank}_{t1}/{t2}"
    return names


def class_margin(frac, labels):
    """Relative distance to the second-closest class centroid (0 = on the boundary, 1 = unambiguous)."""
    H = hellinger(frac); valid = labels >= 0
    ks = sorted(int(c) for c in np.unique(labels[valid]))
    cent = np.vstack([H[labels == c].mean(0) for c in ks])
    D = np.sqrt(((H[:, None, :] - cent[None, :, :]) ** 2).sum(2))
    o = np.argsort(D, axis=1); r = np.arange(len(D))
    d1, d2 = D[r, o[:, 0]], D[r, o[:, 1]]
    return np.where(valid, (d2 - d1) / (d2 + d1 + 1e-12), np.nan)


def select_k(frac, valid, k_range=(3, 4, 5, 6), seed=42, min_ari=0.80, sil_tol=0.02, n_sub=30000):
    """Pick the smallest k that is split-half stable and within sil_tol of the best silhouette."""
    rng = np.random.default_rng(seed); H = hellinger(frac); vidx = np.where(valid)[0]
    sub = rng.choice(vidx, min(n_sub, len(vidx)), replace=False)
    perm = rng.permutation(vidx); third = len(perm) // 3
    ia, ib, ic = perm[:third], perm[third:2 * third], perm[2 * third:]
    rows = []
    for k in k_range:
        km = KMeans(k, n_init=5, random_state=seed).fit(H[sub])
        sil = silhouette_score(H[sub], km.labels_, sample_size=min(10000, len(sub)), random_state=seed)
        a = KMeans(k, n_init=5, random_state=1).fit(H[ia]); b = KMeans(k, n_init=5, random_state=2).fit(H[ib])
        rows.append((k, sil, adjusted_rand_score(a.predict(H[ic]), b.predict(H[ic]))))
    ok = [r for r in rows if r[2] >= min_ari] or rows
    best = max(r[1] for r in ok)
    return int(min(r[0] for r in ok if r[1] >= best - sil_tol)), rows


Writing src/spatialml/targets.py


In [ ]:
%%writefile src/spatialml/demo.py
"""spatialml.demo - a small synthetic tissue so the whole pipeline can run (and be tested) without the real data."""
import numpy as np
import pandas as pd

from .annotation import COARSE_TYPES

_MIX = {  # region -> probability of each coarse type
    0: [.80, .03, .00, .05, .00, .02, .05, .05],
    1: [.60, .15, .00, .05, .00, .03, .07, .10],
    2: [.10, .30, .05, .20, .01, .04, .10, .20],
    3: [.05, .10, .03, .12, .01, .09, .20, .40],
    4: [.15, .05, .00, .10, .00, .10, .25, .35],
    5: [.70, .05, .00, .05, .00, .02, .08, .10],
}


def make_synthetic_tissue(n_cells=6000, density_per_mm2=4000, seed=0):
    """Random cell positions in a square tissue; six Voronoi-like regions with different cell-type mixtures."""
    rng = np.random.default_rng(seed)
    side = float(np.sqrt(n_cells / density_per_mm2) * 1000.0)
    xy = rng.uniform(0, side, (n_cells, 2))
    centers = rng.uniform(0, side, (len(_MIX), 2))
    region = np.argmin(((xy[:, None, :] - centers[None]) ** 2).sum(2), axis=1)
    types = np.empty(n_cells, dtype=object)
    for r, p in _MIX.items():
        m = region == r
        types[m] = rng.choice(COARSE_TYPES, size=int(m.sum()), p=np.array(p) / sum(p))
    return pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "cell_type": types, "region": region})


Writing src/spatialml/demo.py


In [ ]:
%%writefile src/spatialml/pipeline.py
"""spatialml.pipeline - an end-to-end, leakage-safe run on synthetic data (demo and integration test)."""
import json
from pathlib import Path

import numpy as np
import pandas as pd

from . import evaluation as ev
from . import modeling as ml
from . import neighborhood as nbh
from .annotation import COARSE_TYPES
from .config import load_config, set_seed
from .demo import make_synthetic_tissue
from .features import build_spatial_features
from .targets import annulus_composition, class_margin, fit_target_classes, name_classes


def _bal_acc_ci(y, proba, idx, blk, n_boot, rng):
    codes, uni = pd.factorize(blk[idx]); nb = len(uni); K = proba.shape[1]
    Cb = ml.block_confusions(y[idx], np.nan_to_num(proba[idx]).argmax(1), codes, nb, K)
    point = float(ml.metrics_from_counts(Cb.sum(0))["balanced_accuracy"])
    boot = ml.metrics_from_counts(np.einsum("rb,bkl->rkl", ml.bootstrap_weights(nb, n_boot, rng), Cb))["balanced_accuracy"]
    lo, hi = np.nanpercentile(boot, [2.5, 97.5])
    return point, float(lo), float(hi), boot


def run_demo(out_dir, n_cells=6000, seed=0, n_bootstrap=200, block_um=300, config=None):
    """Synthetic tissue -> strict features -> target -> spatial CV -> results. Returns a dict and writes JSON/CSV."""
    cfg = load_config(overrides={"cv": {"block_um": block_um, "folds": 5}, "seed": seed, **(config or {})})
    rng = set_seed(cfg["seed"]); out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)

    tissue = make_synthetic_tissue(n_cells, seed=seed)
    xy = tissue[["x", "y"]].to_numpy(); ct = tissue["cell_type"].to_numpy()
    fc, tc = cfg["features"], cfg["target"]
    feats = build_spatial_features(xy, ct, tuple(fc["radii_um"]), fc["cap_um"], tuple(fc["knn_ks"])).astype(np.float32)
    frac, n_ann, valid = annulus_composition(xy, ct, tuple(tc["annulus_um"]), min_cells=tc["min_cells_in_annulus"])
    labels = fit_target_classes(frac, valid, k=tc["k"], seed=cfg["seed"])
    names = name_classes(frac, labels); margin = class_margin(frac, labels)
    K = len(names); y = labels.copy()

    _, fold = nbh.make_spatial_folds(xy, cfg["cv"]["block_um"], cfg["cv"]["folds"], seed=cfg["seed"])
    blk = pd.factorize(np.floor(xy[:, 0] / cfg["cv"]["block_um"]) * 10000 + np.floor(xy[:, 1] / cfg["cv"]["block_um"]))[0]
    type_cols = [f"type_{t}" for t in COARSE_TYPES + ["Unassigned"]]
    X = feats.copy()
    for t in COARSE_TYPES + ["Unassigned"]:
        X[f"type_{t}"] = (ct == t).astype(np.float32)
    X["x"] = xy[:, 0].astype(np.float32); X["y"] = xy[:, 1].astype(np.float32)
    strict_cols = list(feats.columns)

    spl = ev.spatial_splits(xy, valid, fold, cfg["cv"]["buffer_um"], cfg["evaluation"]["max_train_cells"], cfg["seed"])
    rspl = ev.random_splits(valid, cfg["cv"]["folds"], cfg["evaluation"]["max_train_cells"], cfg["seed"])
    idx = np.where(valid)[0]
    P = {}
    P["own_type_baseline"] = ev.run_cv(X, y, spl, type_cols, "lookup", {"type_cols": type_cols}, cfg["seed"], K)
    P["spatial_strict"] = ev.run_cv(X, y, spl, strict_cols, "hgb", cfg["model"], cfg["seed"], K)
    P["coordinates_spatial_cv"] = ev.run_cv(X, y, spl, ["x", "y"], "hgb", cfg["model"], cfg["seed"], K)
    P["coordinates_random_cv"] = ev.run_cv(X, y, rspl, ["x", "y"], "hgb", cfg["model"], cfg["seed"], K)

    rows = []
    for name, p in P.items():
        pt, lo, hi, _ = _bal_acc_ci(y, p, idx, blk, n_bootstrap, rng)
        rows.append({"experiment": name, "balanced_accuracy": pt, "ci_low": lo, "ci_high": hi})
    res = pd.DataFrame(rows); res.to_csv(out / "demo_results.csv", index=False)
    summary = {"n_cells": int(n_cells), "n_valid": int(valid.sum()), "classes": [names[c] for c in sorted(names)],
               "mean_margin": float(np.nanmean(margin)), "chance": 1.0 / K,
               "results": {r["experiment"]: r["balanced_accuracy"] for r in rows}, "config": cfg}
    json.dump(summary, open(out / "demo_summary.json", "w"), indent=2)
    return summary


Writing src/spatialml/pipeline.py


In [ ]:
%%writefile src/spatialml/cli.py
"""Command line interface: `spatialml demo` runs the synthetic end-to-end pipeline."""
import argparse
import json

from . import __version__
from .config import load_config
from .pipeline import run_demo


def main(argv=None):
    p = argparse.ArgumentParser(prog="spatialml", description="Leakage-safe spatial ML for Xenium data")
    p.add_argument("--version", action="version", version=__version__)
    sub = p.add_subparsers(dest="cmd", required=True)
    d = sub.add_parser("demo", help="run the synthetic end-to-end pipeline")
    d.add_argument("--out", default="demo_output"); d.add_argument("--cells", type=int, default=6000)
    d.add_argument("--seed", type=int, default=0); d.add_argument("--bootstrap", type=int, default=200)
    c = sub.add_parser("check-config", help="validate a configuration file against the leakage rules")
    c.add_argument("path")
    a = p.parse_args(argv)
    if a.cmd == "demo":
        s = run_demo(a.out, a.cells, a.seed, a.bootstrap)
        print(json.dumps({k: v for k, v in s.items() if k != "config"}, indent=2))
    else:
        load_config(a.path); print("configuration OK")


if __name__ == "__main__":
    main()


Writing src/spatialml/cli.py


## Part 3 — Test suite

In [ ]:
%%writefile tests/conftest.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))


Writing tests/conftest.py


In [ ]:
%%writefile tests/test_neighborhood.py
import numpy as np
import pytest
from scipy.spatial import cKDTree

from spatialml import neighborhood as nbh


@pytest.fixture(scope="module")
def tissue():
    rng = np.random.default_rng(0)
    xy = rng.uniform(0, 600, (1500, 2)); codes = rng.integers(0, 4, 1500)
    return xy, codes, np.eye(4, dtype=np.float32)[codes]


def test_ring_stats_matches_brute_force(tissue):
    xy, codes, onehot = tissue
    rings = [(0, 15), (0, 30), (50, 100)]
    st = nbh.ring_stats(xy, onehot, rings, chunk=400)          # small chunk exercises chunk borders
    for i in np.random.default_rng(1).choice(len(xy), 120, replace=False):
        d = np.hypot(*(xy - xy[i]).T)
        for r_in, r_out in rings:
            m = (d >= r_in) & (d <= r_out); m[i] = False         # self is always excluded
            assert st[(r_in, r_out)]["n"][i] == m.sum()
            assert np.allclose(st[(r_in, r_out)]["counts"][i], onehot[m].sum(0))


def test_entropy_bounds_and_nan():
    h = nbh.shannon_entropy(np.array([[5, 0, 0], [1, 1, 1], [0, 0, 0]], float))
    assert np.isclose(h[0], 0) and np.isclose(h[1], np.log(3)) and np.isnan(h[2])


def test_nearest_type_distance(tissue):
    xy, codes, _ = tissue
    d = nbh.nearest_type_distance(xy, codes, 2)
    for i in range(0, 1500, 150):
        m = codes == 2; m[i] = False
        assert np.isclose(d[i], np.hypot(*(xy[m] - xy[i]).T).min())


def test_spatial_folds_buffer_and_balance(tissue):
    xy, _, _ = tissue
    block, fold = nbh.make_spatial_folds(xy, 150, 4, seed=0)
    sizes = np.bincount(fold); assert sizes.max() / sizes.min() < 2.0
    for f in range(4):
        tr, te = nbh.buffered_train_mask(xy, fold, f, 60)
        assert not (tr & te).any()
        dmin, _ = cKDTree(xy[te]).query(xy[tr], k=1)
        assert dmin.min() >= 60 - 1e-9
    for b in np.unique(block):                                   # a block never spans two folds
        assert len(np.unique(fold[block == b])) == 1


def test_nearest_type_distance_with_a_single_cell_of_the_type():
    xy = np.array([[0.0, 0.0], [10.0, 0.0], [0.0, 25.0], [100.0, 100.0]])
    codes = np.array([0, 0, 0, 1])                       # type 1 has exactly one cell
    d = nbh.nearest_type_distance(xy, codes, 1)
    assert np.isnan(d[3]) and np.allclose(d[:3], np.hypot(xy[:3, 0] - 100, xy[:3, 1] - 100))
    assert np.isnan(nbh.nearest_type_distance(xy, codes, 2)).all()                 # absent type -> NaN


Writing tests/test_neighborhood.py


In [ ]:
%%writefile tests/test_stats.py
import numpy as np
import pandas as pd
from scipy.stats import kruskal
from sklearn.metrics import roc_auc_score

from spatialml import stats as st


def _data(n=3000, K=3, seed=0):
    rng = np.random.default_rng(seed)
    y = rng.integers(0, K, n)
    X = pd.DataFrame({"a": rng.normal(y, 1.0), "b": rng.normal(0, 1, n), "c": np.round(rng.normal(y * 0.3, 1, n), 1)})
    X.loc[rng.choice(n, 100, replace=False), "b"] = np.nan                     # NaNs and ties are handled
    return X, y, K


def test_eta2_equals_kruskal_and_auc_equals_sklearn():
    X, y, K = _data()
    P, M = st.rank_matrix(X)
    S, N, Q = st.group_sums(P, M, y, K)
    eta = st.eta2_from_sums(S[0], N[0], Q[0]); auc = st.auc_from_sums(S[0], N[0])
    for j, c in enumerate(X.columns):
        x = X[c].to_numpy(float); ok = ~np.isnan(x)
        H = kruskal(*[x[ok & (y == k)] for k in range(K)]).statistic
        assert np.isclose(eta[j], H / (ok.sum() - 1), rtol=1e-6)
        for k in range(K):
            assert np.isclose(auc[k, j], roc_auc_score((y[ok] == k).astype(int), x[ok]), atol=1e-6)


def test_block_bootstrap_ci_contains_estimate():
    X, y, K = _data(4000)
    blk = np.random.default_rng(1).integers(0, 40, len(y))
    P, M = st.rank_matrix(X)
    S, N, Q = st.group_sums(P, M, y, K, group=blk, n_groups=40)
    eta = st.eta2_from_sums(S.sum(0), N.sum(0), Q.sum(0))
    boot, _ = st.block_bootstrap(S, N, Q, 300, np.random.default_rng(2))
    lo, hi = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    assert np.all(lo <= eta + 1e-9) and np.all(eta <= hi + 1e-9)


def test_benjamini_hochberg_monotone():
    p = np.array([0.001, 0.01, 0.03, 0.04, 0.5]); q = st.benjamini_hochberg(p)
    assert np.all(q >= p) and np.all(np.diff(q[np.argsort(p)]) >= -1e-12) and q.max() <= 1


def test_within_group_permutation_preserves_group_counts():
    rng = np.random.default_rng(0); g = rng.integers(0, 15, 3000); c = rng.integers(0, 4, 3000)
    c2 = st.within_group_permutation(c, g, rng)
    for k in range(15):
        assert np.array_equal(np.bincount(c[g == k], minlength=4), np.bincount(c2[g == k], minlength=4))
    assert (c != c2).mean() > 0.3


def test_shift_null_keeps_label_distribution():
    rng = np.random.default_rng(0)
    xy = rng.uniform(0, 1500, (8000, 2)); lab = (xy[:, 0] > 750).astype(int)
    new = st.shift_null_labels(xy, lab, rng, min_shift_um=500, max_assign_um=30)
    kept = new >= 0
    assert kept.mean() > 0.9 and abs(new[kept].mean() - lab.mean()) < 0.1


Writing tests/test_stats.py


In [ ]:
%%writefile tests/test_modeling.py
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, mean_squared_error, r2_score

from spatialml import modeling as ml


def test_block_count_metrics_match_sklearn():
    rs = np.random.default_rng(0); K = 4
    yt = rs.integers(0, K, 5000); yp = np.where(rs.random(5000) < .6, yt, rs.integers(0, K, 5000)); blk = rs.integers(0, 50, 5000)
    m = ml.metrics_from_counts(ml.block_confusions(yt, yp, blk, 50, K).sum(0))
    assert np.isclose(m["balanced_accuracy"], balanced_accuracy_score(yt, yp))
    assert np.isclose(m["macro_f1"], f1_score(yt, yp, average="macro"))
    assert np.isclose(m["accuracy"], accuracy_score(yt, yp))
    y = rs.random(5000); p = y + rs.normal(0, .2, 5000)
    r2, rmse = ml.r2_rmse_from_sums(ml.block_regression_sums(y, p, blk, 50).sum(0))
    assert np.isclose(r2, r2_score(y, p)) and np.isclose(rmse, mean_squared_error(y, p) ** 0.5)


def test_type_lookup_probabilities_and_prediction():
    cols = ["type_a", "type_b"]
    X = pd.DataFrame({"type_a": [1, 1, 1, 0, 0, 0], "type_b": [0, 0, 0, 1, 1, 1]}); y = np.array([0, 0, 1, 1, 1, 0])
    m = ml.TypeLookup(type_cols=cols).fit(X, y)
    p = m.predict_proba(X)
    assert np.allclose(p.sum(1), 1) and list(m.predict(X)) == [0, 0, 0, 1, 1, 1]


def test_models_fit_predict_with_missing_values():
    rng = np.random.default_rng(0)
    X = pd.DataFrame(rng.normal(size=(600, 6)), columns=["dist_nearest_a", "knn_dist_x", "f1", "f2", "f3", "f4"])
    X[["dist_nearest_a", "knn_dist_x"]] = np.abs(X[["dist_nearest_a", "knn_dist_x"]])      # distances are non-negative
    y = (X["f1"] + X["f2"] > 0).astype(int).to_numpy(); X.iloc[::17, 2] = np.nan
    for kind in ["logreg", "rf", "hgb"]:
        m = ml.build_model(kind, list(X.columns), 0, {"n_estimators": 30, "max_iter": 30}).fit(X, y)
        assert balanced_accuracy_score(y, m.predict(X)) > 0.8


Writing tests/test_modeling.py


In [ ]:
%%writefile tests/test_features.py
import numpy as np
import pytest

from spatialml import features as ft
from spatialml.annotation import COARSE_TYPES
from spatialml.demo import make_synthetic_tissue


@pytest.fixture(scope="module")
def tissue():
    return make_synthetic_tissue(2500, seed=1)


def test_feature_names_and_no_inf(tissue):
    f = ft.build_spatial_features(tissue[["x", "y"]].to_numpy(), tissue["cell_type"].to_numpy())
    assert "frac_Epithelial_r30" in f and "entropy_r15" in f and "knn_dist_cap30_k10" in f
    assert all(f"dist_nearest_{t}_cap30" in f for t in COARSE_TYPES)
    assert np.isfinite(f.to_numpy()[~np.isnan(f.to_numpy())]).all()


def test_capped_distances_never_exceed_cap(tissue):
    f = ft.build_spatial_features(tissue[["x", "y"]].to_numpy(), tissue["cell_type"].to_numpy(), cap_um=30)
    dist_cols = [c for c in f if c.startswith(("dist_nearest_", "knn_dist_"))]
    assert (f[dist_cols].max() <= 30 + 1e-9).all()


def test_reach_guarantee_by_deleting_distant_cells(tissue):
    """Deleting every cell farther than the cap from a focal cell must not change any of its features."""
    worst = ft.reach_test(tissue[["x", "y"]].to_numpy(), tissue["cell_type"].to_numpy(), n_cells=25, seed=0)
    assert worst < 1e-6


def test_radius_above_cap_is_rejected(tissue):
    with pytest.raises(ValueError):
        ft.build_spatial_features(tissue[["x", "y"]].to_numpy(), tissue["cell_type"].to_numpy(), radii=(15, 60), cap_um=30)


def test_density_and_counts_consistent(tissue):
    f = ft.build_spatial_features(tissue[["x", "y"]].to_numpy(), tissue["cell_type"].to_numpy())
    assert np.allclose(f["density_r30"], f["n_cells_r30"] / (np.pi * 0.03 ** 2))
    frac_cols = [c for c in f if c.startswith("frac_") and c.endswith("_r30")]
    s = f[frac_cols].sum(axis=1).to_numpy(); ok = f["n_cells_r30"].to_numpy() > 0
    assert np.allclose(s[ok], 1.0)


Writing tests/test_features.py


In [ ]:
%%writefile tests/test_targets.py
import numpy as np
import pytest

from spatialml import targets as tg
from spatialml.annotation import COARSE_TYPES
from spatialml.config import load_config
from spatialml.demo import make_synthetic_tissue


@pytest.fixture(scope="module")
def data():
    t = make_synthetic_tissue(3500, seed=2)
    xy = t[["x", "y"]].to_numpy(); ct = t["cell_type"].to_numpy()
    frac, n_ann, valid = tg.annulus_composition(xy, ct, (50, 100), min_cells=10)
    return xy, ct, frac, n_ann, valid


def test_annulus_matches_brute_force(data):
    xy, ct, frac, n_ann, valid = data
    for i in np.where(valid)[0][::200]:
        d = np.hypot(*(xy - xy[i]).T); m = (d >= 50) & (d <= 100); m[i] = False
        assert n_ann[i] == m.sum()
        counts = np.array([(ct[m] == t).sum() for t in COARSE_TYPES], float)
        assert np.allclose(frac[i], counts / counts.sum())


def test_classes_names_margin(data):
    xy, ct, frac, n_ann, valid = data
    labels = tg.fit_target_classes(frac, valid, k=3, seed=0)
    assert (labels[~valid] == -1).all() and set(np.unique(labels[valid])) == {0, 1, 2}
    again = tg.fit_target_classes(frac, valid, k=3, seed=0)
    assert np.array_equal(labels, again)                                  # deterministic
    names = tg.name_classes(frac, labels)
    assert len(set(names.values())) == 3 and all(n.startswith("ME") for n in names.values())
    m = tg.class_margin(frac, labels)
    assert np.nanmin(m) >= 0 and np.nanmax(m) <= 1 and np.isnan(m[~valid]).all()


def test_select_k_returns_value_in_range(data):
    _, _, frac, _, valid = data
    k, rows = tg.select_k(frac, valid, k_range=(2, 3, 4), n_sub=2000)
    assert k in (2, 3, 4) and len(rows) == 3


def test_leakage_rules_in_config():
    cfg = load_config()                                                   # defaults obey the rules
    assert cfg["features"]["cap_um"] < cfg["target"]["annulus_um"][0]
    assert cfg["cv"]["buffer_um"] >= cfg["target"]["annulus_um"][1]
    with pytest.raises(ValueError):
        load_config(overrides={"features": {"cap_um": 60, "radii_um": [15, 30]}})     # cap reaches into the target ring
    with pytest.raises(ValueError):
        load_config(overrides={"cv": {"buffer_um": 50}})                              # buffer smaller than target ring


Writing tests/test_targets.py


In [ ]:
%%writefile tests/test_annotation.py
import numpy as np

from spatialml import annotation as an


def _expr(seed=0):
    rng = np.random.default_rng(seed)
    genes = ["A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2"]
    clusters = np.repeat(np.arange(4), 300)
    X = rng.poisson(0.2, (1200, 8)).astype(float)
    X[clusters == 0, 0:2] += 3; X[clusters == 1, 2:4] += 3; X[clusters == 2, 4:6] += 3
    X[clusters == 2, 6:8] += 3              # cluster 2 expresses lineage C AND D equally -> ambiguous
    return np.log1p(X), clusters, genes


def test_annotation_and_ambiguity():
    X, cl, genes = _expr()
    means, sizes = an.cluster_mean_expression(X, cl, genes)
    markers = {"LinA": ["A1", "A2"], "LinB": ["B1", "B2"], "LinC": ["C1", "C2"], "LinD": ["D1", "D2"]}
    table, _ = an.annotate_clusters(means, sizes, markers)
    assert table.loc["0", "label"] == "LinA" and table.loc["1", "label"] == "LinB"
    assert not table.loc["2", "confident"] and table.loc["2", "label"] == "Ambiguous"


def test_refinement_merges_only_compatible_pairs():
    import pandas as pd
    t = pd.DataFrame({"confident": [False, False, True], "label": ["Ambiguous", "Ambiguous", "T_cell"],
                      "top_lineage": ["Macrophage_myeloid", "T_cell", "T_cell"],
                      "second_lineage": ["Dendritic_cell", "Macrophage_myeloid", "B_cell"], "top_score": [0.9, 0.9, 0.9]}, index=["a", "b", "c"])
    r = an.refine_labels(t)
    assert r.loc["a", "fine_label"] == "Myeloid" and r.loc["b", "fine_label"] == "Unassigned" and r.loc["c", "fine_label"] == "T_cell"
    assert list(an.to_coarse(["Myeloid", "Unassigned", "Stromal_adipose"])) == ["Myeloid", "Unassigned", "Stromal"]


Writing tests/test_annotation.py


In [ ]:
%%writefile tests/test_pipeline.py
import json

from spatialml import pipeline


def test_demo_pipeline_runs_and_beats_chance(tmp_path):
    s = pipeline.run_demo(tmp_path, n_cells=3000, seed=0, n_bootstrap=50)
    assert (tmp_path / "demo_results.csv").exists() and (tmp_path / "demo_summary.json").exists()
    r = s["results"]
    assert all(0.0 <= v <= 1.0 for v in r.values())
    assert r["spatial_strict"] > s["chance"] + 0.1                       # real signal exists in the synthetic tissue
    assert r["coordinates_random_cv"] >= r["coordinates_spatial_cv"] - 0.02     # random CV is never harder than spatial CV
    assert json.load(open(tmp_path / "demo_summary.json"))["config"]["features"]["cap_um"] == 30


Writing tests/test_pipeline.py


## Part 4 — Packaging, CI and documentation files

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "spatialml"
version = "1.0.0"
description = "Leakage-safe spatial machine learning for Xenium spatial transcriptomics"
readme = "README.md"
requires-python = ">=3.10"
license = {text = "MIT"}
dependencies = ["numpy>=1.24", "scipy>=1.10", "pandas>=2.0", "scikit-learn>=1.3"]

[project.optional-dependencies]
notebooks = ["anndata>=0.10", "scanpy>=1.10", "h5py", "matplotlib", "seaborn", "pyarrow", "igraph", "ipython", "jupyter"]
dev = ["pytest>=7"]

[project.scripts]
spatialml = "spatialml.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]


Writing pyproject.toml


In [ ]:
%%writefile Makefile
.PHONY: install test demo clean

install:
	pip install -e ".[dev]"

test:
	python -m pytest -q

demo:
	spatialml demo --out demo_output --cells 6000

clean:
	rm -rf demo_output .pytest_cache build *.egg-info


Writing Makefile


In [ ]:
%%writefile .gitignore
# data and large intermediates are never committed
data/
*.h5
*.h5ad
*.parquet
*.npz
ml_checkpoints/
eval_checkpoints/
demo_output/

# python / notebooks
__pycache__/
*.pyc
.ipynb_checkpoints/
*.egg-info/
build/
dist/
.pytest_cache/

# local
.DS_Store
.env
*.zip


Writing .gitignore


In [ ]:
%%writefile .github/workflows/ci.yml
name: tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - name: Install
        run: pip install -e ".[dev]"
      - name: Unit and integration tests
        run: python -m pytest -q
      - name: End-to-end demo on synthetic tissue
        run: spatialml demo --out demo_output --cells 3000 --bootstrap 50


Writing .github/workflows/ci.yml


In [ ]:
%%writefile scripts/download_data.py
"""Download the public 10x Genomics Xenium FFPE Human Breast Cancer (Rep 1) files used in this project.

Usage:  python scripts/download_data.py [--out data/raw]
Check the dataset page on 10xgenomics.com for the current license / terms before redistributing any derived data.
"""
import argparse
import urllib.request
from pathlib import Path

BASE = ("https://cf.10xgenomics.com/samples/xenium/1.0.1/Xenium_FFPE_Human_Breast_Cancer_Rep1/"
        "Xenium_FFPE_Human_Breast_Cancer_Rep1")
FILES = {"cell_feature_matrix.h5": f"{BASE}_cell_feature_matrix.h5", "cells.csv.gz": f"{BASE}_cells.csv.gz"}


def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--out", default="data/raw"); a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    for name, url in FILES.items():
        dest = out / name
        if dest.exists() and dest.stat().st_size > 0:
            print(f"already present: {dest}"); continue
        print(f"downloading {name} ..."); urllib.request.urlretrieve(url, dest)
        if dest.stat().st_size == 0:
            raise RuntimeError(f"empty download: {name}")
        print(f"saved {dest} ({dest.stat().st_size / 1e6:.1f} MB)")


if __name__ == "__main__":
    main()


Writing scripts/download_data.py


In [ ]:
%%writefile docs/methods.md
# Methods

## Question
Does the local cellular neighbourhood of a cell (cells within 30 µm) predict the composition class of the surrounding tissue
(a 50–100 µm ring), beyond what the cell's own transcriptome or cell type tells us?

## Design rules (enforced in code and tests)
1. **Feature reach.** Every feature depends only on cells within `cap_um` (30 µm) of the focal cell; the focal cell is excluded.
   Nearest-type distances are *capped* at 30 µm, because an unbounded distance encodes cells in the target ring.
   `spatialml.features.reach_test` deletes all cells beyond the cap and checks that the features do not change.
2. **Disjoint target.** The target is computed from cells at 50–100 µm only (`spatialml.targets`).
3. **Spatial evaluation.** Tissue is cut into 500 µm blocks assigned to folds. Training cells within 100 µm (the target ring's outer
   radius) of any test cell are removed. Preprocessing is fitted inside each training fold.
4. **Block-level uncertainty.** Confidence intervals resample spatial blocks, not cells. Model comparisons are paired.
5. **Spatial null models.** Feature–class association is compared with a toroidal-shift null that preserves spatial autocorrelation;
   neighbourhood enrichment uses a global and a within-tile (250 µm) label permutation.

## Pipeline
| Step | Code | Notebook |
|---|---|---|
| Data validation and QC | – | 01, 02 |
| Marker-based annotation with uncertainty, label refinement | `spatialml.annotation` | 02, 03 |
| Spatial features | `spatialml.features`, `spatialml.neighborhood` | 03, 06 |
| Target (annulus composition, k-means, margin) | `spatialml.targets` | 03, 04 |
| Statistics and null models | `spatialml.stats` | 04 |
| Models and spatial cross-validation | `spatialml.modeling`, `spatialml.evaluation` | 05, 06 |
| Leakage audit and robustness | `spatialml.features.reach_test`, `spatialml.evaluation` | 06 |

## Leakage found and corrected during the project
An early feature set contained unbounded nearest-type distances. They reached into the target ring and inflated balanced accuracy.
They were replaced by distances capped at 30 µm; the effect is quantified in the report (`reports/scientific_report.md`).
The early results are kept in `results/notebook05_original_features/` for transparency and are **not** the reported results.


Writing docs/methods.md


In [ ]:
%%writefile docs/reproducibility.md
# Reproducibility

## Quick check (2 minutes, no data needed)
```bash
pip install -e ".[dev]"
python -m pytest -q                 # unit and integration tests
spatialml demo --out demo_output    # synthetic end-to-end run
```

## Full analysis (Google Colab, about 3–4 hours in total)
1. `python scripts/download_data.py` (or the download cell in notebook 01).
2. Run the notebooks in `notebooks/` in numeric order, saving to the same project folder (Drive).
3. Long steps (notebooks 05 and 06) write checkpoints; if Colab disconnects, re-run and finished models are loaded from disk.
4. Notebook 07 refactors the code into this package, checks it reproduces the stored features and targets, and writes the report.

## Determinism
* One seed (`configs/default.json`, `seed`) controls sub-sampling, k-means, bootstrap and permutation draws.
* Gradient boosting and k-means are deterministic for a fixed seed and library versions; the random forest uses `n_jobs=-1` but a fixed `random_state`.
* The exact versions used are in `requirements-lock.txt` and `environment.json`.

## Known environment issues
* **Do not downgrade pandas inside Colab notebooks** (e.g. `pip install pandas==2.2.3`). Files written by a newer pandas/anndata cannot always be read by an older one
  (`TypeError: StringDtype.__init__() got an unexpected keyword argument 'na_value'`). The expression loader in notebook 06 reads the HDF5 layer directly to avoid this.
* Colab warns that `google-colab` pins pandas 2.2.x. The package declares support for pandas >= 2.0; it was developed against pandas 3.0, and the test suite (run in notebook 07 and in CI) checks whichever version is installed - read its result rather than assuming.


Writing docs/reproducibility.md


## Part 5 — Run the tests and the synthetic end-to-end demo

In [ ]:
try:
    import pytest  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)

env = dict(os.environ, PYTHONPATH=str(REPO_DIR / "src"))
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-800:] if r.returncode else "")
assert r.returncode == 0, "tests failed - read the output above before continuing"
n_tests = int(re.search(r"(\d+) passed", r.stdout).group(1)); print(f"All {n_tests} tests passed in {time.time()-t0:.0f}s")

.........................                                                [100%]
25 passed in 13.47s


All 25 tests passed in 16s


In [ ]:
out = REPO_DIR / "demo_output"
r = subprocess.run([sys.executable, "-m", "spatialml.cli", "demo", "--out", str(out), "--cells", "6000", "--bootstrap", "200"], cwd=REPO_DIR, env=env, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-1500:]
demo = json.load(open(out / "demo_summary.json"))
print("Synthetic demo (balanced accuracy, spatial CV unless stated):")
for k, v in demo["results"].items(): print(f"  {k:26s} {v:.3f}")
print(f"  chance level               {demo['chance']:.3f}")
print("Expected pattern: spatial_strict > own_type_baseline; coordinates_random_cv >> coordinates_spatial_cv (the leakage demonstration).")

Synthetic demo (balanced accuracy, spatial CV unless stated):
  own_type_baseline          0.500
  spatial_strict             0.826
  coordinates_spatial_cv     0.774
  coordinates_random_cv      0.979
  chance level               0.333
Expected pattern: spatial_strict > own_type_baseline; coordinates_random_cv >> coordinates_spatial_cv (the leakage demonstration).


## Part 6 — Equivalence check: does the library reproduce your stored features and targets?
Refactoring is only trustworthy if the new functions give **the same numbers** as the notebooks. We recompute the strict features, the annulus composition, the class labels (with the same k-means seed) and the class margins from the cell types and coordinates, and compare them with the saved tables from Notebooks 03–04.

In [ ]:
equiv = {}
if RUN_EQUIVALENCE:
    sys.path.insert(0, str(REPO_DIR / "src"))
    for m in [k for k in sys.modules if k.startswith("spatialml")]: del sys.modules[m]
    import spatialml; from spatialml import features as sf, targets as tg
    from sklearn.metrics import adjusted_rand_score

    CFG = json.load(open(PROJECT_DIR / "configs/config.json")); TD = CFG["target_design"]; SEED = CFG["seed"]
    cap = CFG.get("strict_feature_cap_um", 30); radii = tuple(TD["feature_neighborhood_radii_um"]); ta = tuple(TD["target_annulus_um"])
    pq = PROC_DIR / "features_cell_level.parquet"
    T = pd.read_parquet(pq) if pq.exists() else pd.read_csv(PROC_DIR / "features_cell_level.csv.gz", index_col=0)
    xy = T[["x", "y"]].to_numpy(float); ct = T["cell_type"].to_numpy()

    t0 = time.time(); F = sf.build_spatial_features(xy, ct, radii, cap, (5, 10))
    rows = []
    for c in F.columns:
        if c in T.columns: ref = T[c].to_numpy(float)
        elif c.startswith("dist_nearest_") and c.endswith(f"_cap{cap}"):
            ref = np.minimum(T[c[:-len(f"_cap{cap}")]].to_numpy(float), cap); ref = np.where(np.isnan(ref), cap, ref)
        else: continue                                   # capped kNN means were not stored; covered by the reach test
        a = F[c].to_numpy(float); ok = np.allclose(a, ref, rtol=1e-4, atol=1e-4, equal_nan=True)
        rows.append({"feature": c, "matches": ok, "max_abs_diff": float(np.nanmax(np.abs(a - ref))) if np.isfinite(a).any() else 0.0})
    E = pd.DataFrame(rows)
    print(f"Features: {int(E['matches'].sum())}/{len(E)} columns identical to the stored table (max abs diff {E['max_abs_diff'].max():.2e}); {time.time()-t0:.0f}s")
    print("Mismatches:", E.loc[~E["matches"], "feature"].tolist() or "none")

    t0 = time.time(); frac, n_ann, valid = tg.annulus_composition(xy, ct, ta, min_cells=TD["min_cells_in_annulus"])
    same_valid = float((valid == T["target_valid"].astype(bool).to_numpy()).mean())
    K = CFG["notebook03"]["chosen_k"]; labels = tg.fit_target_classes(frac, valid, K, seed=SEED, n_init=10)
    names = tg.name_classes(frac, labels); lab_names = pd.Series(labels).map(names).where(labels >= 0, "insufficient_context").to_numpy()
    v = valid & T["target_valid"].astype(bool).to_numpy()
    same_lab = float((lab_names[v] == T["microenvironment_class"].to_numpy()[v]).mean())
    ari = float(adjusted_rand_score(lab_names[v], T["microenvironment_class"].to_numpy()[v]))
    margin = tg.class_margin(frac, labels); apq = PROC_DIR / "target_annulus_composition.parquet"
    mg = np.nan
    if apq.exists():
        stored = pd.read_parquet(apq)["target_margin"].to_numpy(float); mg = float(np.nanmax(np.abs(margin - stored)))
    print(f"Target: valid-mask agreement {same_valid:.4f} | class-name agreement {same_lab:.4f} | ARI {ari:.4f} | max margin diff {mg:.2e}; {time.time()-t0:.0f}s")
    equiv = {"features_compared": int(len(E)), "features_identical": int(E["matches"].sum()), "valid_mask_agreement": same_valid,
             "class_label_agreement": same_lab, "ARI": ari, "max_margin_abs_diff": None if np.isnan(mg) else mg}
    if not (E["matches"].all() and same_valid > 0.999 and same_lab > 0.999):
        print("WARNING: the library does not exactly reproduce the notebook outputs - inspect before publishing.")
    else:
        print("PASS: the refactored library reproduces the stored features and targets.")
    json.dump(equiv, open(REPO_DIR / "reports/equivalence_check.json", "w"), indent=2)
else:
    print("Equivalence check skipped (RUN_EQUIVALENCE = False)")

Features: 38/38 columns identical to the stored table (max abs diff 9.59e-04); 2s
Mismatches: none
Target: valid-mask agreement 1.0000 | class-name agreement 1.0000 | ARI 1.0000 | max margin diff 0.00e+00; 5s
PASS: the refactored library reproduces the stored features and targets.


## Part 7 — Capture the exact environment

In [ ]:
PKGS = ["numpy", "scipy", "pandas", "scikit-learn", "anndata", "scanpy", "h5py", "pyarrow", "matplotlib", "seaborn", "igraph", "joblib", "pytest", "ipython"]
vers = {}
for p in PKGS:
    try: vers[p] = imd.version(p)
    except imd.PackageNotFoundError: pass
(REPO_DIR / "requirements-lock.txt").write_text("# versions this analysis was run and tested with\n" + "\n".join(f"{k}=={v}" for k, v in vers.items()) + "\n")
envinfo = {"python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count(), "packages": vers,
           "note": "The package declares support for pandas >= 2.0; the test suite was run against the versions listed above. Do not downgrade pandas inside Colab."}
json.dump(envinfo, open(REPO_DIR / "environment.json", "w"), indent=2)
print(json.dumps(envinfo, indent=2))

{
  "python": "3.13.18",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "cpu_count": 2,
  "packages": {
    "numpy": "2.4.6",
    "scipy": "1.16.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "h5py": "3.14.0",
    "pyarrow": "23.0.1",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.2",
    "joblib": "1.6.0",
    "pytest": "8.4.2",
    "ipython": "7.34.0"
  },
  "note": "The package declares support for pandas >= 2.0; the test suite was run against the versions listed above. Do not downgrade pandas inside Colab."
}


## Part 8 — Curate results, copy cleaned notebooks, audit the repository
* **Results:** tables and figures are copied (small files only). Notebook 05's files (`ml_*`) used the *original* features that contained the long-range-distance leakage, so they go into `results/notebook05_original_features/` with a warning, not into the main results.
* **Notebooks:** found in your Drive, outputs cleared, and any `pip install pandas==...` line removed from the copy (it breaks reproducibility).
* **Audit:** large files, tokens/secrets, and hard-coded personal paths.

In [ ]:
MAX_MB = 5
def copy_small(src, dst, max_px=1800):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.suffix == ".png":                      # downscale large figures so the repository stays light
        try:
            from PIL import Image
            im = Image.open(src)
            if im.width > max_px: im = im.resize((max_px, int(im.height * max_px / im.width)), Image.LANCZOS)
            im.save(dst, optimize=True)
            if dst.stat().st_size > 1e6:              # flat-colour figures compress far better with a 256-colour palette
                q = dst.with_suffix(".q.png"); im.convert("RGB").quantize(colors=256).save(q, optimize=True)
                if q.stat().st_size < dst.stat().st_size: q.replace(dst)
                else: q.unlink()
        except Exception:
            shutil.copy(src, dst)
    else:
        shutil.copy(src, dst)
    if dst.stat().st_size > MAX_MB * 1e6:
        print(f"  skipped (> {MAX_MB} MB): {src.name}"); dst.unlink(); return False
    return True

n_main = n_arch = 0
for kind, ext in [("tables", "*.csv"), ("figures", "*.png")]:
    for p in sorted((PROJECT_DIR / f"results/{kind}").glob(ext)):
        if p.name.startswith("ml_"):
            n_arch += copy_small(p, REPO_DIR / f"results/notebook05_original_features/{kind}/{p.name}")
        else:
            n_main += copy_small(p, REPO_DIR / f"results/{kind}/{p.name}")
(REPO_DIR / "results/notebook05_original_features/README.md").write_text(
    "# Notebook 05 results (original feature set)\n\n**These results used unbounded nearest-type distances, which reach into the 50-100 um target ring "
    "and inflate performance (see `reports/scientific_report.md`).** They are kept only to document the leakage and must not be quoted as results. "
    "The corrected, leakage-controlled results are in `results/tables/eval_*.csv` (Notebook 06).\n")
for p in sorted(REP_DIR.glob("*")):
    if p.is_file() and p.suffix in (".json", ".md"): copy_small(p, REPO_DIR / "reports" / p.name)
cfg_used = json.load(open(PROJECT_DIR / "configs/config.json")); json.dump(cfg_used, open(REPO_DIR / "configs/config_used.json", "w"), indent=2)
sys.path.insert(0, str(REPO_DIR / "src"))
from spatialml.config import DEFAULT_CONFIG, save_config
save_config(DEFAULT_CONFIG, REPO_DIR / "configs/default.json")
print(f"Copied {n_main} main result files and {n_arch} archived (Notebook 05) files.")

Copied 58 main result files and 13 archived (Notebook 05) files.


In [ ]:
import nbformat
roots = [PROJECT_DIR, PROJECT_DIR.parent / "Colab Notebooks", Path("/content/drive/MyDrive/Colab Notebooks")]
found = {}
for root in roots:
    if root.exists():
        for p in root.rglob("notebook_0[1-6]*.ipynb"):
            if REPO_DIR in p.parents or ".ipynb_checkpoints" in str(p): continue
            k = p.name[:11]
            if k not in found or p.stat().st_mtime > found[k].stat().st_mtime: found[k] = p
removed = []
for k, p in sorted(found.items()):
    nbk = nbformat.read(p, as_version=4)
    for cell in nbk.cells:
        if cell.cell_type == "code":
            cell.outputs = []; cell.execution_count = None
            keep = []
            for line in cell.source.split("\n"):
                if re.match(r"^\s*[!%]?\s*pip install.*pandas\s*==", line): removed.append((p.name, line.strip()))
                else: keep.append(line)
            cell.source = "\n".join(keep)
    nbk.metadata.pop("widgets", None)
    nbformat.write(nbk, REPO_DIR / "notebooks" / p.name)
print("Notebooks copied (outputs cleared):", [p.name for p in sorted(found.values())] or "NONE FOUND - download them from Colab (File > Download .ipynb) and put them in notebooks/")
for n, l in removed: print(f"  removed pandas pin from {n}: {l}")

Notebooks copied (outputs cleared): ['notebook_01.ipynb', 'notebook_02_annotation_and_target.ipynb', 'notebook_03_spatial_features_and_target.ipynb', 'notebook_04_statistical_testing.ipynb', 'notebook_05_ml_baselines.ipynb', 'notebook_06_leakage_and_robust_evaluation.ipynb']
  removed pandas pin from notebook_03_spatial_features_and_target.ipynb: !pip install -q "pandas==2.1.4" "anndata==0.10.3"


In [ ]:
# ---------------- repository audit ----------------
files = [p for p in REPO_DIR.rglob("*") if p.is_file() and ".git" not in p.parts]
big = [(p, p.stat().st_size / 1e6) for p in files if p.stat().st_size > 50e6]
warn_mid = [(p, p.stat().st_size / 1e6) for p in files if 10e6 < p.stat().st_size <= 50e6]
SECRET = re.compile(r"(ghp_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,}|AKIA[0-9A-Z]{16}|-----BEGIN [A-Z ]*PRIVATE KEY-----|sk-[A-Za-z0-9]{32,})")
secrets, drive_paths = [], 0
for p in files:
    if p.suffix in (".py", ".ipynb", ".md", ".json", ".toml", ".yml", ".txt", ".cff", ".csv"):
        try: txt = p.read_text(errors="ignore")
        except Exception: continue
        if SECRET.search(txt): secrets.append(p)
        if p.suffix == ".py" and "/content/drive" in txt: drive_paths += 1
print(f"Files in repo: {len(files)} | total size {sum(p.stat().st_size for p in files)/1e6:.1f} MB")
print("Files > 50 MB (GitHub rejects > 100 MB):", [(str(p.relative_to(REPO_DIR)), round(s, 1)) for p, s in big] or "none")
print("Files 10-50 MB:", [(str(p.relative_to(REPO_DIR)), round(s, 1)) for p, s in warn_mid] or "none")
print("Possible secrets/tokens:", [str(p.relative_to(REPO_DIR)) for p in secrets] or "none")
print("Library .py files with hard-coded Drive paths:", drive_paths)
assert not big and not secrets, "fix the problems above before publishing"

Files in repo: 142 | total size 9.7 MB
Files > 50 MB (GitHub rejects > 100 MB): none
Files 10-50 MB: none
Possible secrets/tokens: none
Library .py files with hard-coded Drive paths: 0


## Part 9 — Scientific report, README, license and citation (generated from your saved results)
All numbers are read from `eval_*.csv`, the notebook report JSONs and the equivalence check. If a file is missing the text says "n/a" instead of guessing.

In [ ]:
def J(p):
    p = Path(p); return json.load(open(p)) if p.exists() else {}
def Cs(n):
    p = TAB_DIR / n; return pd.read_csv(p) if p.exists() else None

r02, r03, r04, r06 = (J(REP_DIR / f) for f in ["notebook_02_annotation_report.json", "notebook_03_report.json", "notebook_04_report.json", "notebook_06_report.json"])
abl, pc, rnd, buf, part, ring, bnd = (Cs(n) for n in ["eval_feature_set_ablation.csv", "eval_paired_contrasts.csv", "eval_random_vs_spatial_cv.csv",
                                                       "eval_buffer_sensitivity.csv", "eval_partition_sensitivity.csv", "eval_target_ring_sensitivity.csv", "eval_boundary_analysis.csv"])
fsets = J(REP_DIR / "feature_sets_final.json")

def row(key):
    if abl is None: return None
    r = abl[abl["experiment"] == key]; return None if r.empty else r.iloc[0]
def ba(key):
    r = row(key); return "n/a" if r is None else f"{r['balanced_accuracy']:.3f} (95% CI {r['ci_low']:.3f}-{r['ci_high']:.3f})"
def val(key):
    r = row(key); return None if r is None else float(r["balanced_accuracy"])
def ctr(a, b):
    if pc is None: return None
    r = pc[(pc["A"] == a) & (pc["B"] == b)]; return None if r.empty else r.iloc[0]
def cf(a, b):
    r = ctr(a, b); return "n/a" if r is None else f"{r['diff_bal_acc']:+.3f} (95% CI {r['ci_low']:+.3f} to {r['ci_high']:+.3f})"
def verdict(a, b, up, down, none):
    r = ctr(a, b)
    if r is None: return "n/a"
    return up if (r["ci_low"] > 0) else (down if r["ci_high"] < 0 else none)

leak = ctr("spatial_strict", "spatial_original")
leak_confirmed = leak is not None and bool(leak["ci_high"] < 0)                       # strict is significantly LOWER than original
stable = (buf is not None and part is not None and (buf["balanced_accuracy"].max() - buf["balanced_accuracy"].min()) <= 0.02
          and (part["balanced_accuracy"].max() - part["balanced_accuracy"].min()) <= 0.02)
ring_ok = ring is not None and len(ring) > 1 and bool((np.diff(ring["strict"].to_numpy()) < 0).all())
ring_above = ring is not None and bool((ring["strict"] > ring["type_lookup"]).all())
imp = Cs("ml_group_importance.csv"); imp_sent = ""
if imp is not None and len(imp):
    top = imp.sort_values("mean", ascending=False).iloc[0]
    if str(top.iloc[0]).startswith("dist_nearest"):
        imp_sent = f"Feature-importance inspection in the first modelling round showed that nearest-type distances were the most important feature group (permutation drop {top['mean']:.2f}). "
qc = cfg_used.get("qc", {})
n_cells = r02.get("cells_after_qc") or r03.get("n_cells") or "n/a"; n_genes = r02.get("genes", "n/a")
n_strict = len(fsets.get("spatial_strict", [])) or "n/a"
cap = cfg_used.get("strict_feature_cap_um", 30); ring0 = cfg_used["target_design"]["target_annulus_um"]
classes = ", ".join(f"`{c}`" for c in r03.get("target_classes", [])) or "n/a"

names = {"type_lookup": "Own cell type only (cross-validated baseline)", "coordinates": "Coordinates (x, y) only", "long_range_probe": "Long-range distances only (leakage probe)",
         "spatial_original": "Spatial, original (unbounded distances; **leaky**)", "spatial_strict": f"Spatial, strict (reach <= {cap} um)",
         "strict_r15_only": "Strict spatial, 15 um features only", "strict_r30_only": "Strict spatial, 30 um + capped distances",
         "strict_composition": "Strict neighbour composition only", "expression": "Expression only (own transcriptome)",
         "expression_nonmarker": "Expression excluding marker genes", "combined": "Strict spatial + expression"}
tbl = "| Features | Balanced accuracy (95% CI) |\n|---|---|\n" + "\n".join(f"| {nm} | {ba(k)} |" for k, nm in names.items() if row(k) is not None)

rand_txt = "n/a"
if rnd is not None and not rnd.empty:
    rr = rnd.set_index("features")
    rand_txt = "; ".join(f"{f}: {rr.loc[f, 'random_cv']:.3f} (random) vs {rr.loc[f, 'spatial_cv']:.3f} (spatial)" for f in rr.index)
buf_txt = f"{buf['balanced_accuracy'].min():.3f}-{buf['balanced_accuracy'].max():.3f} across buffers of {int(buf['buffer_um'].min())}-{int(buf['buffer_um'].max())} um" if buf is not None else "n/a"
part_txt = f"{part['balanced_accuracy'].min():.3f}-{part['balanced_accuracy'].max():.3f} across block sizes {sorted(int(b) for b in part['block_um'].unique())} um and {part['seed'].nunique()} partition seeds" if part is not None else "n/a"
ring_txt = "; ".join(f"{r.ring_um} um: {r.strict:.3f} (own-type baseline {r.type_lookup:.3f})" for r in ring.itertuples()) if ring is not None else "n/a"
bnd_txt = "n/a"
if bnd is not None and (bnd["model"] == "spatial_strict").any():
    b_ = bnd[bnd["model"] == "spatial_strict"].iloc[0]; bnd_txt = f"{b_['high_margin_90pct']:.3f} on the 90% of cells far from class boundaries vs {b_['low_margin_10pct']:.3f} on the 10% closest to a boundary"
eq_txt = (f"The refactored library reproduces the stored features and targets ({equiv.get('features_identical', 'n/a')}/{equiv.get('features_compared', 'n/a')} feature columns identical; "
          f"class-label agreement {equiv.get('class_label_agreement', float('nan')):.4f})." if equiv else "Equivalence check not run.")

LIM = ["Single tissue section from one patient: confidence intervals reflect variation across tissue blocks, not across patients; generalisation is untested.",
       "Cell types and the target are derived from the same 313-gene expression data, so the own-type baseline and the expression-only model are partly circular with the target; the analysis excluding marker genes mitigates but does not remove this.",
       "Tumour and normal epithelium cannot be separated with this panel without copy-number evidence.",
       "The target is a gradient cut into three classes; cells near class boundaries are classified much less reliably.",
       "Hyper-parameters were tuned by nested spatial cross-validation in Notebook 05 and reused in Notebook 06 without tuning on test folds; training cells were sub-sampled for speed.",
       "An early version of the analysis contained a leakage (unbounded nearest-type distances). It was found by feature-importance inspection, confirmed, and removed; other undetected leakage routes cannot be excluded, which is why the design rules are enforced by tests."]
(REPO_DIR / "reports/limitations.md").write_text("# Limitations\n\n" + "\n".join(f"- {l}" for l in LIM) + "\n")

1068

In [ ]:
leak_sent = (f"An early analysis with unbounded nearest-type distances reached {ba('spatial_original')}; this was identified as leakage (the features reached into the target ring) and corrected (difference {cf('spatial_strict', 'spatial_original')})."
             if leak_confirmed else
             f"A variant with unbounded nearest-type distances reached {ba('spatial_original')} versus {ba('spatial_strict')} for the strict features (difference {cf('spatial_strict', 'spatial_original')}); the strict features are used throughout because they cannot reach into the target ring by construction.")
stab_sent = "Results were stable across buffer size, block size and partition seed" if stable else "Results varied with buffer size, block size or partition seed (Section 6)"
if ring_ok: stab_sent += "; accuracy decreased monotonically as the target ring moved away from the cell" + (" but stayed above the own-type baseline" if ring_above else "")
stab_sent += "."
report = f'''# Predicting tumour microenvironment class from spatial neighbourhoods in Xenium breast cancer data

## Abstract
We asked whether the cellular neighbourhood of a cell (cells within {cap} um) predicts the composition class of the surrounding tissue (a {ring0[0]}-{ring0[1]} um ring), beyond what the cell's own transcriptome or cell type reveals.
Using a public Xenium FFPE human breast cancer section ({n_cells} cells, {n_genes} genes) and spatial block cross-validation with a {cfg_used['target_design']['spatial_cv_buffer_um']} um train/test buffer,
leakage-controlled spatial features reached a balanced accuracy of {ba('spatial_strict')}, versus {ba('type_lookup')} for an own-cell-type baseline and {ba('expression')} for expression alone.
Combining spatial and expression features changed balanced accuracy relative to expression alone by {cf('combined', 'expression')}.
{leak_sent} {stab_sent} All results come from one section and are exploratory.

## 1. Data and quality control
Public 10x Genomics Xenium FFPE Human Breast Cancer (Rep 1). Cell identifiers and coordinates were validated, cells with fewer than {qc.get('min_counts_per_cell', 10)} counts or {qc.get('min_genes_per_cell', 3)} detected genes were removed, and raw counts were kept in a separate layer.

## 2. Cell-type labels
Clusters (Leiden, several resolutions) were scored against marker lists restricted to genes on the panel. A cluster was labelled only if its best lineage score was high and clearly above the runner-up; otherwise it was flagged ambiguous. Ambiguous clusters were resolved only when their top two lineages were a biologically compatible pair (for example macrophage and dendritic cell), and otherwise left unassigned and excluded from targets. Labels were checked for stability across resolutions and seeds and by a classifier that never saw the marker genes. Eight coarse types were used for composition features.

## 3. Features, target and evaluation
* **Features ({n_strict}).** Composition, density, diversity, edge-asymmetry and capped distance features computed from cells within {cap} um, excluding the focal cell. Nearest-type distances were capped at {cap} um.
* **Target.** Composition of assigned cells in the {ring0[0]}-{ring0[1]} um ring, Hellinger-transformed and clustered (k-means, k chosen by stability and silhouette) into {len(r03.get('target_classes', [])) or 'n/a'} classes: {classes}. The class margin (distance to the second-closest centroid) quantifies boundary cells.
* **Evaluation.** Tissue blocks of {cfg_used['target_design']['spatial_cv_block_size_um']} um were assigned to folds; training cells within {cfg_used['target_design']['spatial_cv_buffer_um']} um of any test cell were removed; preprocessing was fitted inside each fold. Confidence intervals resample blocks; model comparisons are paired.

## 4. Results
{tbl}

* **Beyond own cell type.** Strict spatial features versus the own-type baseline: {cf('spatial_strict', 'type_lookup')}.
* **Spatial vs expression.** Spatial minus expression-only: {cf('spatial_strict', 'expression')}. Combined minus expression-only: {cf('combined', 'expression')} - {verdict('combined', 'expression', 'spatial features add information beyond the transcriptome in this section', 'adding spatial features reduced accuracy', 'no detectable gain from adding spatial features')}. Combined minus spatial-only: {cf('combined', 'spatial_strict')}.
* **Not just marker genes.** Expression without the genes used to define cell types: {ba('expression_nonmarker')}.
* **Statistical association.** In Notebook 04, {r04.get('n_features_supported', 'n/a')} of {r04.get('n_features', 'n/a')} features were associated with the class above a spatial (toroidal-shift) null model, whereas a naive test flagged {r04.get('naive_p_below_1e-6', 'n/a')} of {r04.get('n_features', 'n/a')}.

## 5. Leakage control
{imp_sent}Nearest-type distances are unbounded: such a distance can describe cells beyond the {cap} um feature radius, including the target ring. The strict features cap them at {cap} um. Strict versus original features: {ba('spatial_strict')} versus {ba('spatial_original')} (difference {cf('spatial_strict', 'spatial_original')}). A coordinate-only model illustrates the effect of non-spatial validation: {rand_txt}. {eq_txt}

## 6. Robustness
* Buffer size: {buf_txt}.
* Block size and partition seed: {part_txt}.
* Moving the target ring outward (strict spatial features): {ring_txt}.
* Boundary cells: {bnd_txt}.

## 7. Limitations
''' + "\n".join(f"* {l}" for l in LIM) + f'''

## 8. Reproducibility
Code is in `src/spatialml` (installable, {n_tests} tests, CI); `python -m pytest` and `spatialml demo` run without the data. Exact package versions are in `requirements-lock.txt`. See `docs/reproducibility.md`.
'''
(REPO_DIR / "reports/scientific_report.md").write_text(report)
print(report[:3800])

# Predicting tumour microenvironment class from spatial neighbourhoods in Xenium breast cancer data

## Abstract
We asked whether the cellular neighbourhood of a cell (cells within 30 um) predicts the composition class of the surrounding tissue (a 50-100 um ring), beyond what the cell's own transcriptome or cell type reveals.
Using a public Xenium FFPE human breast cancer section (164000 cells, 313 genes) and spatial block cross-validation with a 100 um train/test buffer,
leakage-controlled spatial features reached a balanced accuracy of 0.746 (95% CI 0.733-0.756), versus 0.612 (95% CI 0.602-0.621) for an own-cell-type baseline and 0.731 (95% CI 0.717-0.745) for expression alone.
Combining spatial and expression features changed balanced accuracy relative to expression alone by +0.042 (95% CI +0.035 to +0.049).
An early analysis with unbounded nearest-type distances reached 0.833 (95% CI 0.818-0.846); this was identified as leakage (the features reached into the target ring) and correc

In [ ]:
def tree(root, depth=2, prefix=""):
    items = sorted([p for p in root.iterdir() if p.name not in {".git", "__pycache__", ".pytest_cache", "demo_output"} and not p.name.endswith(".egg-info")], key=lambda p: (p.is_file(), p.name))
    lines = []
    for i, p in enumerate(items):
        lines.append(f"{prefix}{'└── ' if i == len(items)-1 else '├── '}{p.name}{'/' if p.is_dir() else ''}")
        if p.is_dir() and depth > 1: lines += tree(p, depth - 1, prefix + ("    " if i == len(items)-1 else "│   "))
    return lines
tree_txt = "\n".join([f"{REPO_NAME}/"] + tree(REPO_DIR, 2))

figs = [("results/figures/spatial_microenvironment_classes.png", "Microenvironment classes on the tissue"),
        ("results/figures/eval_feature_set_ablation.png", "Feature-set ablation under spatial cross-validation (salmon = reaches into the target ring)"),
        ("results/figures/eval_random_vs_spatial_cv.png", "Random vs spatial cross-validation"),
        ("results/figures/eval_target_ring_sensitivity.png", "Sensitivity to the distance between cell and target ring")]
fig_md = "\n\n".join(f"![{c}]({p})\n*{c}*" for p, c in figs if (REPO_DIR / p).exists())

readme = f'''# Leakage-safe spatial machine learning on Xenium breast cancer data

![tests](https://github.com/{GITHUB_USER}/{REPO_NAME}/actions/workflows/ci.yml/badge.svg)

Can the cellular neighbourhood of a cell predict the type of tissue microenvironment around it, **beyond what its own gene expression tells us**?
This project builds the answer step by step on a public Xenium FFPE breast cancer section ({n_cells} cells, {n_genes} genes) - and focuses on making the answer *defensible*: leakage is audited and tested, evaluation is spatially separated, and uncertainty is estimated at the level of tissue blocks.

## Key results
{tbl}

* Strict spatial features vs the own-cell-type baseline: {cf('spatial_strict', 'type_lookup')} ({verdict('spatial_strict', 'type_lookup', 'spatial features outperform the baseline', 'spatial features are worse than the baseline', 'no detectable difference')}).
* Adding spatial features to expression: {cf('combined', 'expression')} ({verdict('combined', 'expression', 'a detectable gain in this section', 'a decrease', 'no detectable gain')}).
* {('**Leakage caught and fixed:** unbounded nearest-type distances reach into the target ring; capping them at ' + str(cap) + ' um changed balanced accuracy from ' + ba('spatial_original') + ' to ' + ba('spatial_strict') + '.') if leak_confirmed else ('Unbounded vs capped nearest-type distances: ' + ba('spatial_original') + ' vs ' + ba('spatial_strict') + '; capped features are used because they cannot reach the target ring by construction.')}
* {stab_sent.rstrip('.')} (see `reports/scientific_report.md`).

{fig_md}

## Why this repository is different
* **Reach guarantee, tested:** `spatialml.features.reach_test` deletes every cell beyond {cap} um from a focal cell and checks that its features do not change.
* **Rules enforced in code:** feature reach < target ring start; CV buffer >= target ring outer radius (`spatialml.config.validate_config`).
* **Spatial cross-validation with buffer**, block-bootstrap confidence intervals, paired model comparisons, spatial null models.
* **Honest negative results:** the leaky early results are archived and labelled, not hidden.

## Quick start (no data needed)
```bash
git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git && cd {REPO_NAME}
pip install -e ".[dev]"
python -m pytest -q                  # {n_tests} tests
spatialml demo --out demo_output     # synthetic end-to-end run
```

## Reproducing the full analysis
1. `python scripts/download_data.py` (10x Genomics public data; check the dataset page for its license).
2. Run `notebooks/` 01-07 in order in Google Colab (about 3-4 hours; long steps checkpoint to Drive). See `docs/reproducibility.md`.

## Repository structure
```text
{tree_txt}
```

## Documentation
* `reports/scientific_report.md` - concise scientific report
* `docs/methods.md` - design rules and pipeline
* `docs/reproducibility.md` - environment, determinism, known issues
* `reports/limitations.md` - what the results cannot show

## Data
Public 10x Genomics Xenium FFPE Human Breast Cancer (Rep 1). Raw data are **not** included; follow the license/terms on the dataset page.

## Citation and license
See `CITATION.cff`. Code released under the MIT License.
'''
(REPO_DIR / "README.md").write_text(readme)

year = time.strftime("%Y")
(REPO_DIR / "LICENSE").write_text(f"MIT License\n\nCopyright (c) {year} {AUTHOR}\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the \"Software\"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED \"AS IS\", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.\n")
(REPO_DIR / "CITATION.cff").write_text(f"cff-version: 1.2.0\nmessage: If you use this software, please cite it as below.\ntitle: Leakage-safe spatial machine learning on Xenium breast cancer data\nversion: 1.0.0\ndate-released: {time.strftime('%Y-%m-%d')}\nauthors:\n  - name: {AUTHOR}\nrepository-code: https://github.com/{GITHUB_USER}/{REPO_NAME}\nlicense: MIT\n")
(REPO_DIR / "notebooks/README.md").write_text("# Notebooks (run in order, Google Colab)\n\n| # | Notebook | Purpose |\n|---|---|---|\n| 01 | data engineering and QC | download, validate IDs/coordinates, QC |\n| 02 | annotation and target | clustering, marker-based labels with uncertainty |\n| 03 | spatial features and target | neighbourhood features, microenvironment target, spatial folds |\n| 04 | statistical testing | effect sizes, block bootstrap, spatial null models |\n| 05 | ML baselines | model ladder under spatial CV (**uses the original, leaky feature set - see results/notebook05_original_features**) |\n| 06 | leakage and robust evaluation | reach audit, strict features, expression vs spatial, sensitivity |\n| 07 | software and portfolio | this repository, tests, report |\n")
print(readme[:1800]); print("...")

# Leakage-safe spatial machine learning on Xenium breast cancer data

![tests](https://github.com/YOUR_GITHUB_USERNAME/spatial-ml-xenium-breast/actions/workflows/ci.yml/badge.svg)

Can the cellular neighbourhood of a cell predict the type of tissue microenvironment around it, **beyond what its own gene expression tells us**?
This project builds the answer step by step on a public Xenium FFPE breast cancer section (164000 cells, 313 genes) - and focuses on making the answer *defensible*: leakage is audited and tested, evaluation is spatially separated, and uncertainty is estimated at the level of tissue blocks.

## Key results
| Features | Balanced accuracy (95% CI) |
|---|---|
| Own cell type only (cross-validated baseline) | 0.612 (95% CI 0.602-0.621) |
| Coordinates (x, y) only | 0.572 (95% CI 0.536-0.611) |
| Long-range distances only (leakage probe) | 0.825 (95% CI 0.810-0.839) |
| Spatial, original (unbounded distances; **leaky**) | 0.833 (95% CI 0.818-0.846) |
| Spatial, strict (

## Part 10 — Local git commit, zip, and publishing instructions

In [ ]:
def git(*args):
    return subprocess.run(["git", *args], cwd=REPO_DIR, capture_output=True, text=True)

if git("--version").returncode == 0:
    if not (REPO_DIR / ".git").exists(): git("init", "-q"); git("branch", "-M", "main")
    git("add", "-A")
    c = git("-c", f"user.name={AUTHOR}", "-c", f"user.email={AUTHOR_EMAIL}", "commit", "-q", "-m", "Leakage-safe spatial ML on Xenium breast cancer data")
    print("git commit:", (c.stdout + c.stderr).strip() or "done"); print(git("log", "--oneline", "-3").stdout)
else:
    print("git is not available in this runtime; use the zip below.")

zip_path = shutil.make_archive(str(PROJECT_DIR / REPO_NAME), "zip", REPO_DIR)
print(f"\nZip for download: {zip_path} ({Path(zip_path).stat().st_size/1e6:.1f} MB)")
print(f'''
Publish on GitHub (do this on your own machine or with the GitHub CLI):
  1. Create an EMPTY public repository named {REPO_NAME} on github.com (no README, no license).
  2. Unzip the file, then:
       cd {REPO_NAME}
       git remote add origin https://github.com/{GITHUB_USER}/{REPO_NAME}.git
       git push -u origin main
  3. Authenticate with `gh auth login` or a personal access token typed into the terminal prompt.
     NEVER paste a token into a notebook cell or commit it to the repository.
  4. In the repo settings add a description and topics (spatial-transcriptomics, xenium, machine-learning, reproducible-research).
''')

git commit: done
5a4a09b Leakage-safe spatial ML on Xenium breast cancer data


Zip for download: /content/drive/MyDrive/spatial_ml_breast_cancer/spatial-ml-xenium-breast.zip (17.7 MB)

Publish on GitHub (do this on your own machine or with the GitHub CLI):
  1. Create an EMPTY public repository named spatial-ml-xenium-breast on github.com (no README, no license).
  2. Unzip the file, then:
       cd spatial-ml-xenium-breast
       git remote add origin https://github.com/YOUR_GITHUB_USERNAME/spatial-ml-xenium-breast.git
       git push -u origin main
  3. Authenticate with `gh auth login` or a personal access token typed into the terminal prompt.
     NEVER paste a token into a notebook cell or commit it to the repository.
  4. In the repo settings add a description and topics (spatial-transcriptomics, xenium, machine-learning, reproducible-research).



## Final checklist (Module 7 deliverable)
- [ ] All tests pass; the demo shows `spatial_strict` > own-type baseline and random-CV coordinates ≫ spatial-CV coordinates
- [ ] Equivalence check: features identical, class-label agreement ≈ 1.0 (if not, **do not publish** until understood)
- [ ] `reports/scientific_report.md` read in full; edit the wording in your own voice and check every number against your figures
- [ ] The README names Notebook 05 results as archived/leaky; no one can mistake 0.833 for the headline result
- [ ] Notebooks in `notebooks/` have cleared outputs and no `pandas==` pins
- [ ] Audit: no file > 50 MB, no secrets; the raw data are not committed
- [ ] Author name, e-mail and GitHub username replaced; license and citation correct
- [ ] Repository public, CI badge green, topics and description set

**Optional next steps:** a second tissue section or dataset to test generalisation across samples; a Docker image; a short methods preprint.